# ***** Part1: Extract Articles Links 

## *This code is used to Find the list of articles DOIs and links for the related keywords defined below (query) , these links can be used later to extract the requried data from the articles

### Notes: 
- This could be used multiple time to generate the papers of interest for each year. It is advised to test for one year to ensure the required articles are indeed the one you require, after that to search for the rest of the years.
- It requires API and institutional keys (provided by Elseiver) to perform the automatic search, by creating the "config.json" file as below.
- the results will be saved as csv files, therefore when running the code again it will overwrite these file, so look for the word "comment/uncomment" to ensure not to overwrite the files after generated.
- Some cells are tagged with "Optional", meaning they are not part of the analysis, rather they provide extra analysis and information if needed by the user.

### # You have to create the file "config.json" with your keys according to below:

 config = {
     "apikey": "Enter your api key",
  "insttoken": "Enter your institute token here"
     }
     
     
* For how to get API key and institute token see following links:
1.  https://dev.elsevier.com/
2.  https://nonprod-devportal.elsevier.com/tecdoc_api_authentication.html
3.  https://researchguides.smu.edu.sg/api-list/elsevier-api#:~:text=How%20to%20get%20an%20Institutional,tied%20to%20your%20API%20key.


## Input Parameters: Change these parameters according to the articles you are searching for

### Search Parameters

In [1]:
search_year = '2015' ## published years of articles, could be single year '2023' or multiple years '2022-2024'. 
## Note: It is advised to search for one year at a time, with maximum of 3 years because there is a limit on the number of articles retreived from Elsvier.

search_keywords = 'dry reforming methane' ## keywords to look for in the articles
## Note: these keywords were chosen based on experience of the paper titles for our topic.
## Note: DO NOT use words such as "of, in, at, ..etc" only include the main keywords of your topic.

search_sort = 'relevance' ## This parameter to choose how to sort the resulted papers, either by 'relevance' OR 'coverDate'

### change file names to save the results and define prefix for your topic
## define topic prefix to use in files/figures names.
topic_analysis = "DMR" 

## file to save the resutls of part 1 and part 2
filename_filtered = f'{topic_analysis}_{search_year}_three_filters.csv' 

### Filter Process: add keywords below to look for in the title and/or abstract of the articles; 

In [2]:
### the resulted articles will match this criteria. The keywords can include both capital and small letters

## 1. keywords to search in the article title, passed as list or words 
title_keywords = ['dry', 'methane', 'reforming']

title_condition = 'all' # this parameter has two possible values: 
# a. 'all': title must include ALL keywords #### Note: choose 'all' for more specified search of the topic
# b. 'any': the title can have ANY of the specified keywords in the title of the article

## 2. Keywords to search in the articles' keywords
keywords_keywords = ['dry', 'reforming', 'methane', 'dmr', 'drm'] 

## 3. Keywords to search in the abstract
abstract_keywords = ['nickel', 'ni']

## Importing Libraries

In [3]:
#Elsevier Libraries
from elsapy.elsclient import ElsClient
from elsapy.elsprofile import ElsAuthor, ElsAffil
from elsapy.elsdoc import FullDoc, AbsDoc
from elsapy.elssearch import ElsSearch

#Libraries for URL access
import json
import csv
import pprint
import requests
import xmltodict
import urllib3
import numpy as np
import pandas as pd
import re      ### Python has a built-in package called re , which can be used to work with Regular Expressions to look for patterns and words in text

#Import warnings library
import warnings
warnings.filterwarnings("ignore")

In [4]:
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
#xml.etree.ElementTree — The ElementTree XML API
import xml.etree.ElementTree as ET

## Accessing the Links - API Extraction

In [6]:
#Create a num array of 0 to 5900 spaces 100 apart per page. Each page of Search returns 100 entries until 6000 is reached
num = np.linspace(0, 5900, 60, dtype = int)
print(num)
################# Important note: Maximum of 6000 searches are returned on calling the ElsSearch API ##################

[   0  100  200  300  400  500  600  700  800  900 1000 1100 1200 1300
 1400 1500 1600 1700 1800 1900 2000 2100 2200 2300 2400 2500 2600 2700
 2800 2900 3000 3100 3200 3300 3400 3500 3600 3700 3800 3900 4000 4100
 4200 4300 4400 4500 4600 4700 4800 4900 5000 5100 5200 5300 5400 5500
 5600 5700 5800 5900]


In [7]:
### Load configuration
con_file = open("config.json")  ## create this file in the directory according to below with your own api and institute keys
config = json.load(con_file)
con_file.close()

###  Append "apikey" and "insttoken" as suggest in the ElsSearch document into a config file:
#  config = {
#      "apikey": "Enter your api key",
#   "insttoken": "Enter your institute token here"
#      }

In [8]:
client = ElsClient(config['apikey'])   #  your api key
client.inst_token = config['insttoken'] # your institute token here

In [9]:
### the XML_DOI function: save the title, URL and Pub_name of every search results from the xml format of the paper

doi_filename = 'doi_.xml'  #### name of the file used in the loop to store the articles data before extraction
def XML_DOI(link): 
    '''
    The XML_DOI function: save the title, URL and Pub_name of every search results from the xml format of the paper
    '''
    ### defining a header's dictionary to pass through requests
    headers_dict = {"X-ELS-APIKey": config['apikey'], "X-ELS-Insttoken": config['insttoken'], "Accept": "application/xml"}
    
    ### x takes response of the HTTP request, passes link
    x = requests.get(link, headers=headers_dict)
    
    ### Save it as XML file
    with open(doi_filename, 'wb') as f:   
        f.write(x.content)

In [10]:
link_list = []  ## empty list to store the links for articles searching

### Loop to iterate over all papers
for i in range(0, 60):           ## loop 60 (i) times, each time with 100 (count) search results ## based on above note #Maximum of 6000 searches are returned on calling the ElsSearch API
 
    ### start, count, query are a few as name suggests
    start = "https://api.elsevier.com/content/search/sciencedirect?" 
    count = "start=" + str(num[i]) + "&count=100"
   
    #### UPI query can take many arguments.
    query = "&query="+search_keywords 
    
    date = "&date="+search_year   ## To search for specific publication year, range or single year ## Reference: https://dev.elsevier.com/documentation/SCOPUSSearchAPI.wadl
    ### Note: you can search for multiple years together, but it is advised to search for one or two years at a time, since the maximum for the search is 6,000 articles
    
    sortby = "&sortBy="+search_sort ## sorted by relevance to the topic, you can also sort by date #### sort = "&sort=coverDate"
    endapi = f"&apiKey={config['apikey']}&insttoken={config['insttoken']}"
    
    link = start + count + query + date + sortby + endapi  ### create the links
    #print(link)
    link_list.append(link)

In [11]:
# link_list[8]    ### check the link to ensure it matches the required serach criteria

## XML Parsing

In [12]:
info_list = [] ## Creating an empty list to store the info, Rerunning this will delete the data that will be stored further ## Note: Run only once!!

## Call the XML_DOI function ### save the title, URL, 'Pub_date', Pub_name, and DOI of every search results from the xml format of the paper
for j in range(0, 60):
    xmlfile = XML_DOI(link_list[j])
    ## Read the data
    ## <?xml version="1.0" encoding="UTF-8"?>
    tree = ET.parse(doi_filename) ## each article is saved in doi_filename in the XML_DOI() function applied above.
    root = tree.getroot()
    for entry in root.findall('{http://www.w3.org/2005/Atom}entry'):
    
        info_dict = {} ## This will create a dictionary where we will store information about the searches
    
        '''
        CHECK CODE
        url = entry.find('{http://prismstandard.org/namespaces/basic/2.0/}url').text
        title = entry.find('{http://purl.org/dc/elements/1.1/}title').text
        pub_name = entry.find('{http://prismstandard.org/namespaces/basic/2.0/}publicationName').text
        doi = entry.find('{http://prismstandard.org/namespaces/basic/2.0/}doi').text
        #description = entry.find('{http://purl.org/dc/elements/1.1/}description').text
        print(url, title, pub_name, doi)
        print('\n')
        
        '''
    
        info_dict['URL'] = entry.find('{http://prismstandard.org/namespaces/basic/2.0/}url').text
        info_dict['Title'] = entry.find('{http://purl.org/dc/elements/1.1/}title').text
        info_dict['Pub_Name'] = entry.find('{http://prismstandard.org/namespaces/basic/2.0/}publicationName').text
        ### added publication year, month, day
        info_dict['Pub_date'] = entry.find('{http://prismstandard.org/namespaces/basic/2.0/}coverDate').text
        
        doi = entry.find('{http://prismstandard.org/namespaces/basic/2.0/}doi')
        if doi is None:
            info_dict['DOI'] = None
        else:
            info_dict['DOI'] = doi.text
    
        info_list.append(info_dict)
    
    #print(info_list)

In [13]:
type(info_list), len(info_list) 
### check if the info are extracted from the articles (if the results is noneType then no articles are extracted)
### Note: the total length must be less than 6,000 to ensure you extracted all related papers, if not you should decrease the number oof years

(list, 1359)

In [14]:
info_list[1000]['Title'] ### checking some titles we can see that most articles are unrelated, for example it has the sentence "It was observed that due to high methane losses" in them, therefore provided in the search. Thus results filteration is required.

'A study on the catalytic hydrogenation of aldehydes using mayenite as active support for palladium'

In [15]:
info_list[1000]

{'URL': 'https://api.elsevier.com/content/article/pii/S156673671500182X',
 'Title': 'A study on the catalytic hydrogenation of aldehydes using mayenite as active support for palladium',
 'Pub_Name': 'Catalysis Communications',
 'Pub_date': '2015-08-05',
 'DOI': '10.1016/j.catcom.2015.04.028'}

## The results are refined by including the articles with the specified keywords

## **First Refinement: Search the title for the keywords

In [16]:
DOI =  []
Title = []
Pub_name = []
Pub_date = []

for i in range(len(info_list)):
    
    if title_condition == 'all':   ## return the articles only if ALL of the keywords is found in the title
        if all(r.lower() in info_list[i]['Title'].lower() for r in title_keywords):
            DOI.append(info_list[i]['DOI'])
            Title.append(info_list[i]['Title'])
            Pub_name.append(info_list[i]['Pub_Name'])
            Pub_date.append(info_list[i]['Pub_date'])
            
    if title_condition == 'any':    ## return the articles if ANY of the keywords is found in the title
        if any(r.lower() in info_list[i]['Title'].lower() for r in title_keywords): ## search for the specified keywords one by one
            DOI.append(info_list[i]['DOI'])
            Title.append(info_list[i]['Title'])
            Pub_name.append(info_list[i]['Pub_Name'])
            Pub_date.append(info_list[i]['Pub_date'])

## Save the results of the title search as dataframe

In [17]:
df_links =  pd.DataFrame()
df_links['Title'] =  Title
df_links['Pub_name'] = Pub_name
df_links['DOI'] = DOI
df_links['Pub_date'] = Pub_date

df_links.head()  ### you may check some of the titles for validation

,Title,Pub_name,DOI,Pub_date
0,Experimental study of methane dry reforming in...,International Journal of Hydrogen Energy,10.1016/j.ijhydene.2015.09.113,2015-12-14
1,Plasma-catalytic dry reforming of methane in a...,Catalysis Today,10.1016/j.cattod.2015.03.040,2015-11-01
2,Dry Methane Reforming Performance of Ni-based ...,Energy Procedia,10.1016/j.egypro.2015.11.450,2015-11-30
3,Effect of support on the performance of Ni-bas...,Journal of Fuel Chemistry and Technology,10.1016/S1872-5813(15)30040-2,2015-11-30
4,Ni–Al hydrotalcite-like material as the cataly...,Comptes Rendus Chimie,10.1016/j.crci.2015.04.005,2015-11-30


In [18]:
Num_search = len(info_list) ## Articles after searching for query and year
Num_title = df_links['Title'].size ## Articles after searching the titles for the keywords
df_links['Title'].size, len(info_list)  ### compare df_links.size to len(info_list) to see how many papers are found according to your title refining

(50, 1359)

In [19]:
# df_links_test = df_links ## EDIT: for testing, delete after...

## Create papers links

### EDIT: does the .appply methods changes the indecies of the dataframe

In [20]:
### Create string1 and string2 to join doi with institute token to make a single URL
string1 = "https://api.elsevier.com/content/article/doi/" 
string2 = f"?apiKey={config['apikey']}&insttoken={config['insttoken']}"

### Access every DOI in the previous file and append the new URL to another column
df_links['Link'] = df_links['DOI'].apply(lambda x: string1 + str(x) + string2)

df_links.head()

,Title,Pub_name,DOI,Pub_date,Link
0,Experimental study of methane dry reforming in...,International Journal of Hydrogen Energy,10.1016/j.ijhydene.2015.09.113,2015-12-14,https://api.elsevier.com/content/article/doi/1...
1,Plasma-catalytic dry reforming of methane in a...,Catalysis Today,10.1016/j.cattod.2015.03.040,2015-11-01,https://api.elsevier.com/content/article/doi/1...
2,Dry Methane Reforming Performance of Ni-based ...,Energy Procedia,10.1016/j.egypro.2015.11.450,2015-11-30,https://api.elsevier.com/content/article/doi/1...
3,Effect of support on the performance of Ni-bas...,Journal of Fuel Chemistry and Technology,10.1016/S1872-5813(15)30040-2,2015-11-30,https://api.elsevier.com/content/article/doi/1...
4,Ni–Al hydrotalcite-like material as the cataly...,Comptes Rendus Chimie,10.1016/j.crci.2015.04.005,2015-11-30,https://api.elsevier.com/content/article/doi/1...


In [21]:
# df_links['Link'][1], df_links['Link'][2], df_links['Link'][3] ### You may use this to check the links

### The Main results of Part 1: is an excel file contains the interested articles according to the search criteria (title, years, sort) links...(later we will use these links to access the articles abstracts/texts).

### Note: Part 1 should be executed more than once if articles for more years are needed

### Final Note: You may compare results for the number of articles with the search on the following website (use the same title and years and sort criteria, ..etc) to validate your code. 
"https://www.sciencedirect.com/search?qs=dry%20reforming%20of%20methane%20nickel&date=2019"

## ---------------------------- End of Part 1 ----------------------------

## *****Part 2: Abstract Extraction

## **This code is used to extract the abstracts of each of the articles along with (DOI, Title, publication date), then saves the abstract to csv file.

## Extract the abstracts (and the keywords) for the articles (Comment/Uncomment)

In [22]:
# df_links['Link'][2] ## EDIT: for testing, delete after...

In [23]:
# df_links ## EDIT: for testing, delete after...

In [24]:
### Comment when the absrtacts are saved

%time
list_abstract = []
list_keywords = []

## ScienceDirect (full-text) document example using DOI
from pprint import pprint
doi= df_links['DOI']          ## used to extract the abstract and the keywords

for i in range(df_links.shape[0]):
    pprint(doi[i])
    doi_doc = FullDoc(doi = doi[i]) ## article text extracted using elaspy see "https://github.com/ElsevierDev/elsapy/tree/master"
    if doi_doc.read(client):

        abstract = doi_doc._data['coredata']['dc:description']                   
        print(i, 'done')
        list_abstract.append(abstract)

        ######## to extract the keywords 
        list_keywords_article = []
        if doi_doc._data['coredata'].get('dcterms:subject') != None: ## Note: not all papers have keywords, thus this condition
            for word in doi_doc._data['coredata']['dcterms:subject']:
                list_keywords_article.append(list(word.values())[1])
            list_keywords.append(list_keywords_article)
        else:
            list_keywords.append(None) ## to include None values where the article has no keywords published by the author

    else:
        pprint('Operation failed') ### if no data were found for the article
        list_abstract.append('0')
        list_keywords.append('0')

df_links['Abstract'] = list_abstract
df_links['Keywords'] = list_keywords

# df_links.head(10)

Wall time: 0 ns
'10.1016/j.ijhydene.2015.09.113'
0 done
'10.1016/j.cattod.2015.03.040'
1 done
'10.1016/j.egypro.2015.11.450'
2 done
'10.1016/S1872-5813(15)30040-2'
3 done
'10.1016/j.crci.2015.04.005'
4 done
'10.1016/j.jiec.2015.08.021'
5 done
'10.1016/j.cej.2014.12.035'
6 done
'10.1016/j.cej.2014.11.109'
7 done
'10.1016/j.apcata.2015.06.009'
8 done
'10.1016/j.jechem.2015.06.015'
9 done
'10.1016/j.apcata.2014.10.026'
10 done
'10.1016/j.apcata.2015.02.019'
11 done
'10.1016/j.rser.2015.02.026'
12 done
'10.1016/j.apcatb.2015.04.039'
13 done
'10.1016/j.apcatb.2015.05.013'
14 done
'10.1016/j.ijhydene.2015.01.005'
15 done
'10.1016/j.cej.2015.06.027'
16 done
'10.1016/j.apcata.2015.01.029'
17 done
'10.1016/j.fuproc.2015.02.023'
18 done
'10.1016/j.cattod.2014.07.054'
19 done
'10.1016/j.crci.2015.03.008'
20 done
'10.1016/j.catcom.2014.09.050'
21 done
'10.1016/j.apcata.2015.07.012'
22 done
'10.1016/j.apcatb.2015.05.018'
23 done
'10.1016/j.enconman.2015.04.053'
24 done
'10.1016/j.ijhydene.2015.09.1

In [25]:
df_links.columns ## check columns
# df_links.head()

Index(['Title', 'Pub_name', 'DOI', 'Pub_date', 'Link', 'Abstract', 'Keywords'], dtype='object')

## **Second Refinement: Search the articles keywords for the keywords (about DMR)

### double check, EDIT: how many articles are left

In [26]:
index_list_keywords = [] ## empty list to save the indices based on the abstract_keywords
for i in range(df_links.shape[0]):
    if isinstance(df_links.iloc[i]['Keywords'], list): ## check if keywords exist
        sentence = ' '.join(word.lower() for word in df_links.iloc[i]['Keywords'])    ### combine keywords into sentence for easier search
        if any(x.lower() in sentence for x in keywords_keywords):
            index_list_keywords.append(i)
    else:
        print('keywords not found')
# print(index_list_keywords)
# print(len(index_list_keywords))

keywords not found


In [27]:
## Apply second refinement
df_links = df_links.iloc[index_list_keywords] ## keep only rows with if interested keywords are found.
Num_keywords = df_links.shape[0] ## Articles after searching the keywords for the keywords
df_links.shape ### check the shape after the second refinement

(46, 7)

## **Third Refinement: Search the abstract for the keywords  

In [28]:
index_list_abstract = [] ## empty list to save the indices based on the abstract_keywords
for i in range(df_links.shape[0]):
    if isinstance(df_links.iloc[i]['Abstract'], str):
        words = re.findall(r'\w+', df_links.iloc[i]['Abstract'], flags=re.IGNORECASE)    ### text is converted to lower case
        for word in words:
            if word.lower() in [x.lower() for x in abstract_keywords]:
                index_list_abstract.append(i)
                break ## the loop find every keyword in the abstract, beacuse multiple words can be found in the abstract, therefore break is used when at least one word is found
# print(index_list_abstract) 
# print(len(index_list_abstract))

In [29]:
## Apply Third refinement
df_links = df_links.iloc[index_list_abstract] ## keep only rows with if interested keywords are found.
Num_abstract = df_links.shape[0] ## Articles after searching the abstracts for the keywords
df_links.shape ### check the shape after the third refinement

(35, 7)

## Save the results of parts 1 and 2 to a csv file (Comment/Uncomment)

In [30]:
df_links.to_csv(filename_filtered, index=True) #### comment if already saved

In [31]:
print(f"The search results of the {topic_analysis} for the year(s) {search_year} are: {Num_search}")
print(f"after the first refinement (title) using the condition {title_condition}, it reduced to: {Num_title}, using the keywords {title_keywords}" )
print(f"after the second refinement (keywords), it reduced to: {Num_keywords}, using the keywords {keywords_keywords}" )
print(f"after the third refinement (abstract), it reduced to: {Num_abstract}, using the keywords {abstract_keywords}" )

The search results of the DMR for the year(s) 2015 are: 1359
after the first refinement (title) using the condition all, it reduced to: 50, using the keywords ['dry', 'methane', 'reforming']
after the second refinement (keywords), it reduced to: 46, using the keywords ['dry', 'reforming', 'methane', 'dmr', 'drm']
after the third refinement (abstract), it reduced to: 35, using the keywords ['nickel', 'ni']


## Main result: Extract the abstracts of the interesteed articles, by ***three stages*** of refinment of the results as keywords in: 
1. Title, 
2. Article keywords, 
3. Abstract.

## The result of this code is an excel file with (n) rows for articles and 

## (  DOI ----- Title --- Abstract --- publication date ---  keywords  ) for each article, as columns.

## ---------------------------- End of Part 2 ----------------------------